In [2]:
import pandas as pd
import json
import requests

In [3]:
response = requests.get('https://firestore.googleapis.com/v1/projects/oeskb-app/databases/(default)/documents/saisonen/1_8_2024_2025/schnittliste/1_8_2024_2025H')
data = response.json()
print(json.dumps(data, indent=2))

{
  "name": "projects/oeskb-app/databases/(default)/documents/saisonen/1_8_2024_2025/schnittliste/1_8_2024_2025H",
  "fields": {
    "spiele": {
      "mapValue": {
        "fields": {
          "nKW1HmZZWos39zWU3lhW": {
            "mapValue": {
              "fields": {
                "heim": {
                  "mapValue": {
                    "fields": {
                      "teamName": {
                        "stringValue": "KSV Wien"
                      },
                      "vereinsNr": {
                        "stringValue": "19113"
                      },
                      "vereinsName": {
                        "stringValue": "KSV Wien"
                      },
                      "spieler": {
                        "mapValue": {
                          "fields": {
                            "HFdZhWlRyDDgubstJkvF": {
                              "mapValue": {
                                "fields": {
                                  "gegner": {
    

In [4]:
results = data['fields']['spiele']['mapValue']['fields']
game_keys = list(results.keys())

result_data = pd.DataFrame()

for game in game_keys:
    for location in ['heim', 'gast']:

        players = results[game]['mapValue']['fields'][location]['mapValue']['fields']['spieler']['mapValue']['fields']
        player_keys = list(players.keys())

        for player_key in player_keys:
            player = players[player_key]['mapValue']['fields']
            player_series = pd.json_normalize(player)
            player_series['spieler_id'] = player_key
            player_series['heim'] = location == 'heim'
            player_series['spiel_id'] = game
            result_data = pd.concat([result_data, player_series], ignore_index=True)

result_data

,gegner.stringValue,kegel.integerValue,mp.integerValue,akJahr.integerValue,land.stringValue,name.stringValue,bahn.integerValue,dg.integerValue,akHJ.integerValue,sp.integerValue,spieler_id,heim,spiel_id,mp.doubleValue,sp.doubleValue
0,KOZAK Jennifer (607),539,0,2002,AUT,196408 / NIKOLIC Maja Katharina,1,1,1,0,HFdZhWlRyDDgubstJkvF,True,nKW1HmZZWos39zWU3lhW,NaN,NaN
1,BAUMGARTNER Mia-Laura (534),550,1,2001,CZE,196074 / PYCHOVA Barbora,1,3,2,2,ThGHwnyCP2jPnqAHQ9yA,True,nKW1HmZZWos39zWU3lhW,NaN,NaN
2,STEINER Fiona (586),588,0,1996,AUT,196054 / LAIBACHER Marina,3,3,2,1,yn7rbBEZboGdDw5IfFoM,True,nKW1HmZZWos39zWU3lhW,NaN,NaN
3,NGUYEN Monika (558),568,0,2006,AUT,196411 / STÖCKL Sandra,3,2,2,1,Sm7t0cDy51j4jJuGBKDi,True,nKW1HmZZWos39zWU3lhW,NaN,NaN
4,HARI Boglarka (648),608,0,1991,AUT,196070 / ANGENBAUER Angelika,3,1,2,0,6tFoI6m2UVaXWO2kYZsm,True,nKW1HmZZWos39zWU3lhW,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
535,LENGAUER Monika (592),543,0,1969,AUT,176026 / HAUSEGGER Andrea,4,1,2,2,1y5AJRx3YjRBg2ZkgmSx,False,BSUfDoHFLTlluYWcQRdl,NaN,NaN
536,MEISTERL Jasmin (586),615,1,2002,AUT,176089 / GSTREIN Franziska,4,2,1,2,2uIGy2DPMwkwdTUMeGvE,False,BSUfDoHFLTlluYWcQRdl,NaN,NaN
537,HAUER Dominique (639),573,0,1999,AUT,176022 / GRUBER Melissa,4,3,1,1,pkZhTZE5WaGWAjCJqRjm,False,BSUfDoHFLTlluYWcQRdl,NaN,NaN
538,EBNER Jessica (541),538,0,1999,ITA,176013 / RABANSER Vera,2,1,2,2,jEdXWyukzwSmGbMHu4Kx,False,BSUfDoHFLTlluYWcQRdl,NaN,NaN


In [5]:
player_data = result_data.drop_duplicates(subset=['spieler_id'])

player_data = player_data.rename(columns={
    'land.stringValue': 'land',
    'name.stringValue': 'name',
    'akJahr.integerValue': 'jahrgang',
})

player_data = player_data[["spieler_id", "name", "jahrgang", "land"]]
player_data[['passnummer', 'name']] = player_data['name'].str.split(' / ', n=1, expand=True)

player_data.to_csv('player.csv', index=False, sep=';')

In [ ]:
# if integer value is not set, take double value
result_data['sp'] = result_data['sp.integerValue'].fillna(result_data['sp.doubleValue'])
result_data['mp'] = result_data['mp.doubleValue'].fillna(result_data['mp.integerValue'])

results = result_data[["spieler_id", "spiel_id", "kegel.integerValue", "sp", "mp", "heim", "bahn.integerValue", "dg.integerValue"]]
results = results.rename(columns={
    'kegel.integerValue': 'kegel',
    'bahn.integerValue': 'startbahn',
    'dg.integerValue': 'dg',
})

In [7]:
response = requests.get('https://firestore.googleapis.com/v1/projects/oeskb-app/databases/(default)/documents/saisonen/1_8_2024_2025')
data = response.json()
print(json.dumps(data, indent=2))

{
  "name": "projects/oeskb-app/databases/(default)/documents/saisonen/1_8_2024_2025",
  "fields": {
    "sportJahr": {
      "stringValue": "2024_2025"
    },
    "teams": {
      "mapValue": {
        "fields": {
          "3": {
            "mapValue": {
              "fields": {
                "vereinsName": {
                  "stringValue": "KSV Wien"
                },
                "id": {
                  "stringValue": "3"
                },
                "name": {
                  "stringValue": "KSV Wien"
                },
                "lvId": {
                  "stringValue": "19"
                },
                "vereinsNr": {
                  "stringValue": "113"
                }
              }
            }
          },
          "4": {
            "mapValue": {
              "fields": {
                "vereinsNr": {
                  "stringValue": "138"
                },
                "name": {
                  "stringValue": "BSV Voith St. P\u00

In [8]:
games = data['fields']['spiele']['mapValue']['fields']
game_keys = list(games.keys())

game_data = pd.DataFrame()

for game_key in game_keys:
    game = games[game_key]["mapValue"]["fields"]["infos"]["mapValue"]["fields"]
    game_series = pd.json_normalize(game)
    game_series['spiel_id'] = game_key
    game_data = pd.concat([game_data, game_series], ignore_index=True)

game_data = game_data.rename(columns={
    'datum.stringValue': 'datum',
    'uhrzeit.stringValue': 'uhrzeit',
    'bahn.stringValue': 'bahn',
})

game_data

game_data = game_data[["datum", "uhrzeit", "bahn", "spiel_id"]]

# concat game data with results
game_data = pd.merge(results, game_data , on='spiel_id', how='left')
game_data

# results.to_csv('game.csv', index=False, sep=';')

,spieler_id,spiel_id,kegel,sp,mp,heim,startbahn,dg,datum,uhrzeit,bahn
0,HFdZhWlRyDDgubstJkvF,nKW1HmZZWos39zWU3lhW,539,0,0,True,1,1,2024-09-28,13:45,KSV Rustenschacherallee
1,ThGHwnyCP2jPnqAHQ9yA,nKW1HmZZWos39zWU3lhW,550,2,1,True,1,3,2024-09-28,13:45,KSV Rustenschacherallee
2,yn7rbBEZboGdDw5IfFoM,nKW1HmZZWos39zWU3lhW,588,1,0,True,3,3,2024-09-28,13:45,KSV Rustenschacherallee
3,Sm7t0cDy51j4jJuGBKDi,nKW1HmZZWos39zWU3lhW,568,1,0,True,3,2,2024-09-28,13:45,KSV Rustenschacherallee
4,6tFoI6m2UVaXWO2kYZsm,nKW1HmZZWos39zWU3lhW,608,0,0,True,3,1,2024-09-28,13:45,KSV Rustenschacherallee
...,...,...,...,...,...,...,...,...,...,...,...
535,1y5AJRx3YjRBg2ZkgmSx,BSUfDoHFLTlluYWcQRdl,543,2,0,False,4,1,2024-11-24,11:00,VSE Sportplatz
536,2uIGy2DPMwkwdTUMeGvE,BSUfDoHFLTlluYWcQRdl,615,2,1,False,4,2,2024-11-24,11:00,VSE Sportplatz
537,pkZhTZE5WaGWAjCJqRjm,BSUfDoHFLTlluYWcQRdl,573,1,0,False,4,3,2024-11-24,11:00,VSE Sportplatz
538,jEdXWyukzwSmGbMHu4Kx,BSUfDoHFLTlluYWcQRdl,538,2,0,False,2,1,2024-11-24,11:00,VSE Sportplatz
